# CutMix Augmentation on Relabeled Dataset

**Hypothesis:** CutMix previously failed because `other` was in the training set — blending images with a visually incoherent class created noise. Now we're training 5-class only (excluding `other`) on clean labels, so all classes have coherent visual identities. CutMix should work better here.

**Setup:** DINOv2 unfreeze-1 + 5-class + confidence thresholding (same as our best config)

**Baseline to beat:** Macro F1: 0.8992 (unfreeze-1, no CutMix)

---

## 1. Setup

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Data

In [ ]:
TRAIN_DIR = './data_train'
TEST_DIR = './data_test'

CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(CLASS_NAMES)
CLEAN_CLASS_NAMES = [c for c in CLASS_NAMES if c != 'other']
NUM_CLEAN = len(CLEAN_CLASS_NAMES)

print(f'All classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Clean classes ({NUM_CLEAN}): {CLEAN_CLASS_NAMES}')

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 30
PATIENCE = 7

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
# 5-class filtered dataset
full_train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
full_test_dataset  = datasets.ImageFolder(root=TEST_DIR,  transform=test_transform)

other_class_idx = full_train_dataset.class_to_idx['other']
clean_indices = [i for i, (_, label) in enumerate(full_train_dataset.samples) if label != other_class_idx]
clean_subset = Subset(full_train_dataset, clean_indices)

old_to_new = {}
new_idx = 0
for old_idx, cls_name in enumerate(CLASS_NAMES):
    if cls_name != 'other':
        old_to_new[old_idx] = new_idx
        new_idx += 1

new_to_old = {v: k for k, v in old_to_new.items()}

class RemappedDataset(torch.utils.data.Dataset):
    def __init__(self, subset, label_map):
        self.subset = subset
        self.label_map = label_map
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        image, label = self.subset[idx]
        return image, self.label_map[label]

clean_train_dataset = RemappedDataset(clean_subset, old_to_new)
clean_train_loader = DataLoader(clean_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=0, pin_memory=True)
full_test_loader = DataLoader(full_test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

print(f'5-class train: {len(clean_train_dataset)} images, {len(clean_train_loader)} batches')
print(f'6-class test:  {len(full_test_dataset)} images')

## 3. Model: DINOv2 Unfreeze-1

In [ ]:
dinov2_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
print(f'DINOv2 ViT-S loaded. Feature dim: {dinov2_backbone.embed_dim}')

In [ ]:
class DINOv2PartialFinetune(nn.Module):
    def __init__(self, backbone, num_classes, unfreeze_last_n=1, embed_dim=384):
        super().__init__()
        self.backbone = backbone
        num_blocks = len(backbone.blocks)
        
        for param in self.backbone.parameters():
            param.requires_grad = False
        for i in range(num_blocks - unfreeze_last_n, num_blocks):
            for param in self.backbone.blocks[i].parameters():
                param.requires_grad = True
        for param in self.backbone.norm.parameters():
            param.requires_grad = True
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.3),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

## 4. CutMix Implementation

In [ ]:
def rand_bbox(size, lam):
    """Generate random bounding box for CutMix."""
    H, W = size[2], size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_h = int(H * cut_rat)
    cut_w = int(W * cut_rat)
    cy = np.random.randint(H)
    cx = np.random.randint(W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    return y1, y2, x1, x2


def apply_cutmix(images, labels, alpha=1.0):
    """Apply CutMix: paste a random patch from one image onto another."""
    lam = np.random.beta(alpha, alpha)
    batch_size = images.size(0)
    index = torch.randperm(batch_size).to(images.device)
    y1, y2, x1, x2 = rand_bbox(images.size(), lam)
    images[:, :, y1:y2, x1:x2] = images[index, :, y1:y2, x1:x2]
    lam = 1 - ((y2 - y1) * (x2 - x1) / (images.size(-1) * images.size(-2)))
    return images, labels, labels[index], lam


def cutmix_criterion(criterion, outputs, labels_a, labels_b, lam):
    """Blended loss for CutMix samples."""
    return lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)

print('CutMix functions defined.')

## 5. Training Functions

In [ ]:
def train_one_epoch_cutmix(model, loader, criterion, optimizer, device, cutmix_prob=0.5):
    """Train with CutMix applied with given probability."""
    model.train()
    running_loss = 0.0; correct = 0; total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        if np.random.rand() < cutmix_prob:
            images, labels_a, labels_b, lam = apply_cutmix(images, labels)
            optimizer.zero_grad()
            outputs = model(images)
            loss = cutmix_criterion(criterion, outputs, labels_a, labels_b, lam)
            loss.backward(); optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (lam * predicted.eq(labels_a).sum().item()
                        + (1 - lam) * predicted.eq(labels_b).sum().item())
        else:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward(); optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, correct / total


def train_one_epoch_normal(model, loader, criterion, optimizer, device):
    """Train without CutMix (for baseline comparison)."""
    model.train()
    running_loss = 0.0; correct = 0; total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, correct / total

In [ ]:
@torch.no_grad()
def predict_with_threshold(model, loader, threshold, clean_class_names, all_class_names, device):
    """5-class model → 6-class predictions via confidence thresholding."""
    model.eval()
    other_idx = all_class_names.index('other')
    
    n2o = {}
    ci = 0
    for oi, cls in enumerate(all_class_names):
        if cls != 'other':
            n2o[ci] = oi
            ci += 1
    
    all_preds, all_labels, all_confs = [], [], []
    for images, labels in loader:
        logits = model(images.to(device))
        probs = torch.softmax(logits, dim=1)
        max_p, pred_5 = probs.max(dim=1)
        
        for i in range(len(labels)):
            c = max_p[i].item()
            all_confs.append(c)
            all_labels.append(labels[i].item())
            all_preds.append(other_idx if c < threshold else n2o[pred_5[i].item()])
    
    return np.array(all_preds), np.array(all_labels), np.array(all_confs)

## 6. Experiment: Compare CutMix Probabilities

We test different CutMix application rates alongside a no-CutMix baseline. All use the same unfreeze-1 architecture and threshold evaluation.

In [ ]:
def run_experiment(cutmix_prob, backbone, loader, test_loader,
                   clean_class_names, all_class_names, device):
    """Train with given CutMix probability, find best threshold, return results."""
    label = f'CutMix {cutmix_prob:.0%}' if cutmix_prob > 0 else 'No CutMix (baseline)'
    print(f'\n{"="*60}')
    print(f'EXPERIMENT: {label}')
    print(f'{"="*60}')
    
    model = DINOv2PartialFinetune(backbone, len(clean_class_names), unfreeze_last_n=1).to(device)
    criterion = nn.CrossEntropyLoss()
    
    backbone_params = list(model.backbone.blocks[11].parameters()) + \
                      list(model.backbone.norm.parameters())
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': 1e-5},
        {'params': model.classifier.parameters(), 'lr': 1e-3},
    ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=1, eta_min=1e-6)
    
    best_acc = 0.0; patience_counter = 0
    save_name = f'best_cutmix_{int(cutmix_prob*100)}.pth'
    
    print(f'\n{"Epoch":>6} | {"Loss":>8} | {"Acc":>8}')
    print('-' * 30)
    
    for epoch in range(1, NUM_EPOCHS + 1):
        if cutmix_prob > 0:
            loss, acc = train_one_epoch_cutmix(
                model, loader, criterion, optimizer, device, cutmix_prob=cutmix_prob)
        else:
            loss, acc = train_one_epoch_normal(
                model, loader, criterion, optimizer, device)
        scheduler.step()
        
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), save_name)
            patience_counter = 0
        else:
            patience_counter += 1
        
        if epoch % 5 == 0 or epoch == 1:
            print(f'{epoch:>6} | {loss:>8.4f} | {acc:>8.4f}')
        
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break
    
    print(f'Best train acc: {best_acc:.4f}')
    
    # Load best and sweep thresholds
    model.load_state_dict(torch.load(save_name, map_location=device))
    
    best_threshold = 0.5; best_f1 = 0.0
    for t in np.arange(0.40, 0.95, 0.05):
        preds, labels, _ = predict_with_threshold(
            model, test_loader, t, clean_class_names, all_class_names, device)
        mf1 = f1_score(labels, preds, average='macro')
        if mf1 > best_f1:
            best_f1 = mf1; best_threshold = t
    
    # Final evaluation
    preds, labels, confs = predict_with_threshold(
        model, test_loader, best_threshold, clean_class_names, all_class_names, device)
    accuracy = (preds == labels).mean()
    
    print(f'\nThreshold: {best_threshold:.2f}')
    print(f'Macro F1:  {best_f1:.4f}')
    print(f'Accuracy:  {accuracy:.4f}')
    print(classification_report(labels, preds, target_names=all_class_names, digits=4))
    
    return {
        'cutmix_prob': cutmix_prob,
        'label': label,
        'train_acc': best_acc,
        'threshold': best_threshold,
        'macro_f1': best_f1,
        'accuracy': accuracy,
        'preds': preds,
        'labels': labels,
    }

In [ ]:
# Run experiments: baseline + 3 CutMix rates
results = []

for prob in [0.0, 0.3, 0.5, 0.7]:
    result = run_experiment(
        cutmix_prob=prob,
        backbone=dinov2_backbone,
        loader=clean_train_loader,
        test_loader=full_test_loader,
        clean_class_names=CLEAN_CLASS_NAMES,
        all_class_names=CLASS_NAMES,
        device=device,
    )
    results.append(result)
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 7. Results Comparison

In [ ]:
comp_df = pd.DataFrame([{
    'Experiment': r['label'],
    'Train Acc': f"{r['train_acc']:.4f}",
    'Threshold': f"{r['threshold']:.2f}",
    'Macro F1': f"{r['macro_f1']:.4f}",
    'Accuracy': f"{r['accuracy']:.4f}",
} for r in results])

print('=' * 75)
print('CUTMIX EXPERIMENT COMPARISON')
print('=' * 75)
print(comp_df.to_string(index=False))
print('=' * 75)

best_result = max(results, key=lambda r: r['macro_f1'])
print(f'\nBest: {best_result["label"]} (Macro F1: {best_result["macro_f1"]:.4f})')

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

probs_list = [r['cutmix_prob'] for r in results]
f1_list = [r['macro_f1'] for r in results]
acc_list = [r['accuracy'] for r in results]
train_acc_list = [r['train_acc'] for r in results]

# F1 and Accuracy by CutMix probability
x = np.arange(len(results))
w = 0.35
axes[0].bar(x - w/2, f1_list, w, label='Macro F1', color='#4CAF50')
axes[0].bar(x + w/2, acc_list, w, label='Accuracy', color='#2196F3')
axes[0].set_xticks(x)
axes[0].set_xticklabels([r['label'] for r in results], rotation=15, ha='right', fontsize=8)
axes[0].set_ylabel('Score')
axes[0].set_title('Test Performance by CutMix Rate', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0.7, 1.0)
axes[0].grid(axis='y', alpha=0.3)
for i, (f1v, accv) in enumerate(zip(f1_list, acc_list)):
    axes[0].text(i - w/2, f1v + 0.005, f'{f1v:.4f}', ha='center', fontsize=8)
    axes[0].text(i + w/2, accv + 0.005, f'{accv:.4f}', ha='center', fontsize=8)

# Train acc vs Test F1 — checking for overfitting
axes[1].plot(probs_list, train_acc_list, 'bo-', label='Train Acc', ms=8)
axes[1].plot(probs_list, f1_list, 'gs-', label='Test Macro F1', ms=8)
axes[1].set_xlabel('CutMix Probability')
axes[1].set_ylabel('Score')
axes[1].set_title('Train vs Test — Overfitting Check', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cutmix_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Best Model — Detailed Evaluation

In [ ]:
best = best_result
preds = best['preds']
labels = best['labels']

print(f'Best: {best["label"]}, threshold={best["threshold"]:.2f}')
print(f'Macro F1:  {best["macro_f1"]:.4f}')
print(f'Accuracy:  {best["accuracy"]:.4f}')
print(f'\n{"="*70}')
print('CLASSIFICATION REPORT')
print(f'{"="*70}')
print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(labels, preds)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Confusion Matrix (Recall)', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_cutmix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Full Project Comparison

In [ ]:
full_comparison = pd.DataFrame({
    'Experiment': [
        'EfficientNet-B0 + imbalance mitigation',
        'EfficientNet-B0 baseline',
        'DINOv2 6-class (before relabeling)',
        'DINOv2 + CutMix 6-class (before relabeling)',
        'DINOv2 6-class (after relabeling)',
        'DINOv2 frozen + threshold',
        'DINOv2 unfreeze-1 + threshold',
        'DINOv2 unfreeze-1 + k-NN ensemble',
        f'DINOv2 unfreeze-1 + {best["label"]}',
    ],
    'Macro F1': [
        0.7645, 0.8089, 0.8244, 0.7896,
        0.8261, 0.8628, 0.8992,
        0.8813, best['macro_f1'],
    ],
    'Accuracy': [
        0.7667, 0.7833, 0.8167, 0.7833,
        0.8333, 0.8500, 0.9000,
        0.8833, best['accuracy'],
    ],
    'Test Set': [
        'Original', 'Original', 'Original', 'Original',
        'Relabeled', 'Relabeled', 'Relabeled',
        'Relabeled', 'Relabeled',
    ]
})

print('=' * 95)
print('FULL PROJECT COMPARISON')
print('=' * 95)
print(full_comparison.to_string(index=False))
print('=' * 95)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
y = np.arange(len(full_comparison))
w = 0.35

ax.barh(y + w/2, full_comparison['Macro F1'], w, label='Macro F1', color='#4CAF50')
ax.barh(y - w/2, full_comparison['Accuracy'], w, label='Accuracy', color='#2196F3')

ax.set_yticks(y)
ax.set_yticklabels(full_comparison['Experiment'], fontsize=8)
ax.set_xlabel('Score')
ax.set_title('Full Project — Experiment Comparison', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
ax.set_xlim(0.6, 1.0)
ax.grid(axis='x', alpha=0.3)

for i, (f1v, accv) in enumerate(zip(full_comparison['Macro F1'], full_comparison['Accuracy'])):
    ax.text(f1v + 0.003, i + w/2, f'{f1v:.4f}', va='center', fontsize=7)
    ax.text(accv + 0.003, i - w/2, f'{accv:.4f}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig('full_comparison_cutmix.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary

### Why re-test CutMix?
CutMix previously failed on the 6-class model because blending images with `other` (visually incoherent) created noise. Now we're training 5-class only on clean, relabeled data — every class has a coherent visual identity, which is exactly the condition CutMix needs to work.

### What was tested:
- No CutMix (baseline)
- CutMix at 30%, 50%, 70% application rate
- All using DINOv2 unfreeze-1 + confidence thresholding

### Key question:
Does CutMix provide meaningful regularization benefit when all training classes are visually clean and coherent?

Check the comparison table above for the answer.